# Human data example

The ancestral alleles most widely used for human data are those of the ancestral sequence distributed by Ensembl, reconstructed by the Enredo–Pecan–Ortheus (EPO) pipeline ([Paten et al., 2008](https://doi.org/10.1101/gr.076521.108)) from an alignment of ten primate reference genomes on a fixed species tree. This guide runs {class}`~ancestree.inference.FixedTreeInference` and {class}`~ancestree.local_tree_inference.LocalTreeInference` on human genomes and the same primate outgroups and compares their assignments with the ancestral sequence. `msl.vcf.gz` covers chr1:95–100 Mb (GRCh38). It holds 10 phased diploid individuals of the Mende in Sierra Leone (MSL) sample of the 1000 Genomes Project ([1000 Genomes Project Consortium, 2015](https://doi.org/10.1038/nature15393)), the ingroup of 20 haplotypes, and the nine other primates of that alignment as outgroups, each as a haploid sample that is missing where its genome is not aligned to the human reference.

In [1]:
from cyvcf2 import VCF

outgroups = ["chimpanzee", "bonobo", "gorilla", "orangutan", "gibbon", "rhesus",
             "cynomolgus", "vervet", "mouse_lemur"]
ids = [s for s in VCF("msl.vcf.gz").samples if s not in outgroups]
print(f"{len(ids)} ingroup individuals, {len(outgroups)} outgroups")

10 ingroup individuals, 9 outgroups


## Model inputs

Both inference modes use HKY with the transition/transversion ratio $\kappa$ and the stationary base frequencies $\pi = (\pi_A, \pi_C, \pi_G, \pi_T)$ estimated over the accessible positions. `msl.bed` lists the accessible positions, at which every haplotype of the sample is called.

In [2]:
import numpy as np
import ancestree as anc

model = anc.HKY(kappa=4.16)
pi = [0.3177, 0.1805, 0.1804, 0.3214]
access = np.loadtxt("msl.bed", usecols=(1, 2), dtype=np.int64)
n_access = int(np.diff(access).sum())
composition = anc.BaseComposition.from_n_target_sites(n_access, stationary=pi)
print(f"{n_access:,} accessible positions")

4,013,074 accessible positions


## Fixed-tree mode

An {class}`~ancestree.trees.OutgroupLadderTree` joins each outgroup to the ingroup lineage at a node of its own, so it holds one outgroup per successively more distant lineage. Bonobo, cynomolgus and vervet share their lineage with chimpanzee or rhesus and are thus left out. {class}`~ancestree.inference.FixedTreeInference` fits the branch rates of the ladder by maximum likelihood and reports, for each site, the posterior probability of each base being ancestral at the human–chimpanzee ancestor, the first node of the ladder above the ingroup and the node of the ancestral sequence (see {doc}`focal_node`).

In [3]:
ladder = ["chimpanzee", "gorilla", "orangutan", "gibbon", "rhesus", "mouse_lemur"]
fixed_tree = anc.Inference.from_fixed_tree(
    source="msl.vcf.gz", ingroup_samples=ids, outgroup_samples=ladder, model=model,
    base_composition=composition, n_target_sites=n_access,
    focal=anc.FocalNode("ingroup_mrca", coalescences=1), n_starts=1,
)
fixed_tree.fit()
res_fixed = fixed_tree.infer()

INFO:ancestree.CyVCF2Source: Reading variants from msl.vcf.gz (38 haplotype samples)
INFO:ancestree.FixedTreeInference: Fitting 11 branch rate(s) by ML over the config histogram.
INFO:ancestree.FixedTreeInference: L-BFGS-B converged in 17 iterations (marginal log-likelihood -9711622.6462)
INFO:ancestree.FixedTreeInference: MLE branch rates: K0=5.0326e-03, K1=5.7940e-03, K2=1.4792e-03, K3=7.2133e-03, K4=7.3821e-03, K5=1.4760e-02, K6=2.0907e-03, K7=1.6657e-02, K8=9.8517e-03, K9=2.9701e-02, K10=1.3322e-01
INFO:ancestree.FixedTreeInference: Ingroup MRCA → outgroup divergences: chimpanzee_h0=1.0827e-02, gorilla_h0=1.3725e-02, orangutan_h0=2.8654e-02, gibbon_h0=3.2641e-02, rhesus_h0=5.5538e-02, mouse_lemur_h0=1.5906e-01


## Local-tree mode

{class}`~ancestree.local_tree_inference.LocalTreeInference` takes all nine outgroups as tips, each a haploid sample in a diploid file. Besides the accessibility mask ({paramref}`accessibility <ancestree.local_tree_inference.LocalTreeInference.accessibility>`), it reads the mutation map ({paramref}`mutation_map <ancestree.local_tree_inference.LocalTreeInference.mutation_map>`) `msl_mu.bedgraph.gz`, the germline mutation rate ([Seplyarskiy et al., 2023](https://doi.org/10.1038/s41588-023-01562-0)) scaled to a mean of $1.25 \times 10^{-8}$ per site per generation over the accessible positions, and the recombination map ({paramref}`recombination_map <ancestree.local_tree_inference.LocalTreeInference.recombination_map>`) `msl_rec.txt`, the HapMap II map of the region ([International HapMap Consortium, 2007](https://doi.org/10.1038/nature06258)). It reports the posterior at the human–chimpanzee ancestor, as fixed-tree mode does.

In [4]:
local_tree = anc.Inference.from_local_tree(
    source="msl.vcf.gz", model=model, mu=1.25e-8, rec_rate=1e-8, base_composition=composition,
    ingroup_samples=ids, outgroup_samples=outgroups,
    accessibility="msl.bed", mutation_map="msl_mu.bedgraph.gz",
    recombination_map="msl_rec.txt", n_ensemble=64,
    focal=anc.FocalNode("ingroup_mrca", coalescences=1),
)
res_local = local_tree.infer()

INFO:ancestree.CyVCF2Source: Reading variants from msl.vcf.gz (38 haplotype samples)
INFO:ancestree.LocalTreeInference: Leaving out 9 haplotype(s) that no call reaches, such as the second slot of a haploid sample in a diploid file: chimpanzee_h1, bonobo_h1, gorilla_h1, orangutan_h1, gibbon_h1, rhesus_h1, cynomolgus_h1, vervet_h1, mouse_lemur_h1


## Comparison with EPO

The `INFO/AA` field of `msl.vcf.gz` holds the base of the ancestral sequence at each site. We grade both inference modes against this base with a {class}`~ancestree.posterior.Grade` over the sites polymorphic within the sample ({class}`~ancestree.sites.PolymorphicSiteFilter`). It reports the fraction of sites whose MAP allele is this base and the mean Brier score of the posterior against it, which decreases as the posterior concentrates on that base.

In [5]:
epo = {record.POS: record.INFO["AA"].upper() for record in VCF("msl.vcf.gz")}
polymorphic = anc.PolymorphicSiteFilter(samples=ids)
grades = {"fixed-tree": anc.Grade(res_fixed, epo, filter=polymorphic),
          "local-tree": anc.Grade(res_local, epo, filter=polymorphic)}
print(f"{'mode':<30} {'sites':>6} {'MAP agreement':>14} {'mean Brier':>11}")
for label, g in grades.items():
    print(f"{label:<30} {g.n_sites:>6,} {g.map_recovery:>14.1%} {g.brier:>11.4f}")

INFO:ancestree.FixedTreeInference: Using 10 ingroup sample(s): HG03052, HG03054, HG03055, HG03057, HG03058, HG03060, HG03061, HG03063, HG03064, HG03066
INFO:ancestree.FixedTreeInference: Using 6 outgroup sample(s): chimpanzee_h0, gorilla_h0, orangutan_h0, gibbon_h0, rhesus_h0, mouse_lemur_h0
INFO:ancestree.FixedTreeInference: Inferring the ancestral allele on the fitted outgroup ladder, streaming the sites
FixedTreeInference: 843513 sites [00:32, 25924.52 sites/s]
INFO:ancestree.FixedTreeInference: baseline (MajorityOutgroupInference) MAP agreement: 0.967 on 100000 polymorphic sites; largest disagreement at SFS bin 3 (0.960); SFS bin 0 (0.967) (leading 100,000-site sample)
INFO:ancestree.FixedTreeInference: Focal node: reported at "ingroup_mrca" (coalescences=1)
INFO:ancestree.LocalTreeInference: Inferring local trees from genotypes (29 samples, window=8snp)
INFO:ancestree.LocalTreeInference: Using 20 ingroup sample(s): HG03052_h0, HG03052_h1, HG03054_h0, HG03054_h1, HG03055_h0, HG0305

mode                            sites  MAP agreement  mean Brier
fixed-tree                     13,999          99.5%      0.0088
local-tree                     13,999          99.4%      0.0103


In [6]:
fixed, local = grades.values()
assert fixed.map_recovery > 0.98
assert local.map_recovery > 0.98